<a href="https://colab.research.google.com/github/SotaYoshida/Lecture_DataScience/blob/master/Kadai_Y.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 課題Y

提出方法はJupyterNotebookに課題に相当する部分を付け足して提出しても良いし、
保存した図をワードか何かに貼り付けて整形しても良い.


---

以下の工程に沿って吉田が作成した例:
https://drive.google.com/file/d/15P_NKAScAa9_-wG7mKpzNIelFY86m2Mq/view?usp=sharing
のようなバブルチャートを何通りか作成しながら、劇的な変化(推移)を示す項目・国などに注目して、その背景を推測する。  


---




##### データの取得について

https://www.gapminder.org/data/
にアクセスして興味のあるデータを選ぶと、
csvファイルやエクセルがダウンロードできる。


**補足**  
本来ならWebから自動に取得したエクセルファイルを開いて
リストにデータを突っ込んで解析するところまでPythonですべてやるのが楽ちんです。  
授業では時間の都合上扱えませんでしたが、スクレイピング用のライブラリやエクセルシートを扱うためのライブラリなどもあるので、興味のある人は調べてみましょう。  




取得したデータを、GoogleColab上の適当な名前のフォルダに入れる。

まずGoogleドライブをマウントして

In [0]:
from google.colab import drive
drive.mount('./gdrive')               

Drive already mounted at ./gdrive; to attempt to forcibly remount, call drive.mount("./gdrive", force_remount=True).


マイドライブのすぐ下に、Colab_csvという名前のフォルダを作ります

In [0]:
!mkdir  'gdrive/My Drive/Colab_csv' #一度実行すればOK  

mkdir: cannot create directory ‘gdrive/My Drive/Colab_csv’: File exists


このフォルダの中に、ダウンロードしたファイルをおいておきましょう。

以下では例として、

一人あたりGDP(インフレ補正済)データ    income_per_person_gdppercapita_ppp_inflation_adjusted.csv
と、  

２５歳から34際の女性が就学した平均年数
mean_years_in_school_women_percent_men_25_to_34_years.csv

を使うことにします。

マウントが成功した場合、以下のコードでフォルダの中にあるファイルのリストを表示できます。

In [0]:
!ls  './gdrive/My Drive/Colab_csv/' 

co2_emissions_tonnes_per_person.xlsx
energy_production_total.xlsx
income_per_person_gdppercapita_ppp_inflation_adjusted.csv
mean_years_in_school_women_percent_men_25_to_34_years.csv
population_total.csv


この中から、解析したいものを読み込みます
データを一気に扱うのに、pandasというライブラリを使います。

In [0]:
file1="income_per_person_gdppercapita_ppp_inflation_adjusted.csv" ##興味のあるデータ１つめのファイル名に変更
file2="mean_years_in_school_women_percent_men_25_to_34_years.csv" ##興味のあるデータ2つめのファイル名に変更

import pandas as pd
df = pd.read_csv('./gdrive/My Drive/Colab_csv/'+file1)
df = df.set_index('country')

三行目では国名で検索ができるようにしておきました。



In [0]:
cols = df.columns ## 列名の取得

In [0]:
 ## 興味のある国のリストをつくっておく
target = [ "Japan", "China", "United States", "North Korea", "Sweden",
          "United Kingdom","Egypt", "India", "South Korea", "Sudan"]

#そうすると、該当する国のデータだけを取り出せる。
subdf= df.loc[target]
#print(subdf)

もう一つのデータも同様に取得しておきましょう。

In [0]:
df2 = pd.read_csv('./gdrive/My Drive/Colab_csv/'+file2)
df2 = df2.set_index('country')
subdf2= df2.loc[target]
#print(subdf2)

人口のデータも読み込んでおきましょう

In [0]:
file_p="population_total.csv"
dfp = pd.read_csv('./gdrive/My Drive/Colab_csv/'+file_p)
dfp = dfp.set_index('country')
subdfp= dfp.loc[target]
#print(subdfp)

GDPのデータには予測を含めた1800-2040のデータが、人口のデータには1800-2100のデータがありましたが、
就学年数のデータには1970-2015しかないようです。  
全てのデータがある期間だけを調べることにしましょう。

調べる年をyearsという変数に突っ込んでおきます。

In [0]:
years = list(df2.columns) ## 一番データがある期間の小さいdf2.columnsで列名(西暦)を取得してリストyearsに格納する

さて、これで興味のあるデータ２つと、各国の人口が用意できました。
あとは、年ごとにループを回して、絵を描いてみましょう。

In [0]:
from matplotlib import pyplot as plt 
!pip install japanize-matplotlib 
import japanize_matplotlib 
import numpy as np
from matplotlib import colors as mcolors
#!mkdir  'gdrive/My Drive/Colab_pic/Kadai_Y' #一度実行すればOK 

##色のリストを作っておく
colors = [] 
for i, (cname,color) in enumerate(mcolors.TABLEAU_COLORS.items()):
    colors += [color]
#### TABLEAUは10色なので、10カ国以上塗り分ける場合は他の方法が良い。

### １つ１つpdfで保存したい場合
for tyear in years:
    fig = plt.figure(figsize=(5,5))
    ax = fig.add_subplot(1,1,1)  
    ax.set_facecolor("#D3DEF1") #背景色をつける
    ax.set_xlabel("GDP per capita (infation adjusted)")
    ax.set_ylabel("Mean year in school women/men (24 to 35) [%]")
    ax.set_title(tyear)
    ax.grid(True,axis="both",color="w", linestyle="dotted", linewidth=0.8)
    ax.set_xlim([0,100000]); ax.set_ylim([0,106]) ##横軸と縦軸の範囲を固定
    for i, tc in enumerate(target): ## 国ごとの操作を指定するループ
        x = subdf.loc[tc].at[tyear] # 元のdf.loc[tc].at[tyear]でも可
        y = subdf2.loc[tc].at[tyear] 
        z = subdfp.loc[tc].at[tyear]  
        ts= z/500000  ###適当な数で人口を割って調整
        tcolor = colors[i]  
        ax.scatter(x,y,marker="o",s=ts,color=tcolor,zorder=20000,alpha=0.5)
        ax.text(x,y,tc,color=tcolor,zorder=23000,alpha=0.7,fontsize=5)   
#   plt.savefig("./gdrive/My Drive/Colab_pic/Kadai_Y/GDPvsWomenInSchool_"+tyear+".pdf")  #ベクタ形式がよければ
    plt.savefig("./gdrive/My Drive/Colab_pic/Kadai_Y/GDPvsWomenInSchool_"+tyear+".png",dpi=300) 
    plt.close()

In [0]:
### PNGをまとめてgif(パラパラ漫画のようなもの)で保存したい場合
from PIL import Image
import glob

files = sorted(glob.glob('./gdrive/My Drive/Colab_pic/Kadai_Y/GDPvsWomen*.png')) ##まとめたいpngをワイルドカードで指定
images = list(map(lambda file: Image.open(file), files))
##gifに出力する
images[0].save('./gdrive/My Drive/Colab_pic/Kadai_Y/GDPvsWomen.gif', save_all=True, append_images=images[1:], duration=400, loop=0) 